In [1]:
### Import packages
import anndata as ad
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq
import squidpy as sq
import matplotlib as mpl
from matplotlib import rc_context
import random
import dask
dask.config.set({"dataframe.query-planning": True})
 
import warnings
warnings.filterwarnings("ignore") 

# Note that BANKSY itself is deterministic, here the seeds affect the umap clusters and leiden partition
seed = 1234
np.random.seed(seed)
random.seed(seed)


/home/nnataren/miniforge3/envs/banksy/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


In [ ]:
# ---------------------------------------------------------------------------- #
#                              LOCAL TESTING BLOCK                             #
# ---------------------------------------------------------------------------- #

## Set the dataset_name and related settings to use during this analysis
dataset_name = "GR_lung_non_res_roi" # sample name
pc_label = "35" # Label for the number of principal components used for the purpose of filenames
pc_dims = [int(pc_label)] # The number of principal components stored a list for analyses
lambda_label = "0.20" # File name label for Lambda setting, see comment below. 
lambda_list = [float(lambda_label)] # Lambda setting to tune BANKSY clustering, lambda = 0 is non-spatial, 0.2 is for cell typing, 0.8 if for domain segmentation 
res_label = "0.50" # BANKSY clustering resolution label for resolution chosen to produce plots
resolutions = [float(res_label)] # BANSY can take a list of resolutions and perform clustering at each which is saved in the BANKSY dictionary

nbr_weight_decay = "scaled_gaussian" # This parameter dictates how much neighbouring cells impact to the neighbourhood expression calculations. Using scaled gaussian, the 
# close neigbours contribute more and this decays as you move out to cells further away in the neighbourhood window. It is scaled for local cell density so that weighting doesn't change
# across regions if cells are pack more closely or loosely in different regions
coord_keys = ('x', 'y', 'xy') # Keys to specify coordinate indexes in the anndata Object


In [ ]:
# ---------------------------------------------------------------------------- #
#                                   SET PATHS                                  #
# ---------------------------------------------------------------------------- #
## Set file paths and read in xenium data

## Create a base path
base_dir = "data/xenium"
#base_dir = "/home/nnataren/Documents/PhD/Bioinformatics/Banksy_py_fork/Banksy_py/hpc"

## Create a path to the raw data e.g., unprocessed anndata files, if it does not already exist
raw_path = os.path.join(base_dir, "raw_data")

if not os.path.isdir(raw_path):
    os.makedirs(raw_path)
    print(f"Directory '{raw_path} successfully.")
    
else:
    print(f"Directory '{raw_path} exists.")

## Create path for processed data e.g., the pre-clustered but unfiltered anndata files, if it does not already exist
processed_path = os.path.join(base_dir, "processed", f"{dataset_name}")

if not os.path.isdir(processed_path):
    os.makedirs(processed_path)
    print(f"Directory '{processed_path}' created successfully.")
    
else:
    print(f"Directory '{processed_path}' already exists.")

## Create a path for output data, if it does not already exist
output_path = os.path.join(base_dir, "output", f"{dataset_name}")

if not os.path.isdir(output_path):
    os.makedirs(output_path)
    print(f"Directory '{output_path}' created successfully.")
    
else:
    print(f"Directory '{output_path}' already exists.")

## Create a path for QC results, if it does not already exist
qc_path = os.path.join(base_dir, "output", "QC_testing", f"{dataset_name}")

if not os.path.isdir(qc_path):
    os.makedirs(qc_path)
    print(f"Directory '{qc_path}' created successfully.")
else:
    print(f"Directory '{qc_path}' already exists.")

In [ ]:
file_path =f"{raw_path}/{dataset_name}/{dataset_name}_clustered_spatial_pc{pc_label}_nc{lambda_label}_r{res_label}.h5ad"

print("Exists?", os.path.exists(file_path))

In [ ]:
# -------------------- Read in the processed AnnData file -------------------- #
import gzip
import pickle

adata_lab = ad.read_h5ad(os.path.join(raw_path, f"{dataset_name}", f"{dataset_name}_clustered_spatial_pc{pc_label}_nc{lambda_label}_r{res_label}.h5ad"))
print(f"Pre-lablled anndata object for {dataset_name} sucessfully read in.")

## Read in the banksy_dict dictionary as a .pkl file
## Use gzip to extract compressed dictionary

dict_name = f"{dataset_name}_pc{pc_label}_nc{lambda_label}_r{res_label}_banksy_dict.pkl.gz"

with gzip.open(os.path.join(raw_path, f"{dataset_name}", dict_name), "rb") as f:
    banksy_dict = pickle.load(f)

## Read in the results_df data frame as a .pkl file
results_name = f"results_df_{dataset_name}_pc{pc_label}_nc{lambda_label}_r{res_label}.pkl.gz"

with gzip.open(os.path.join(raw_path, f"{dataset_name}", results_name), "rb") as f:
    results_df = pickle.load(f)